# 3 1 count analysis daily

Peer-review copy. Inputs are pseudonymized processed data; see README.md and DATA_DICTIONARY.md. Original preparation/collection steps are outside this package.

In [ ]:
from review_setup import prepare_workspace
prepare_workspace()
from review_setup import initialize_stata
initialize_stata()


In [ ]:
import pandas as pd
from utils import get_source_corrections, StataDiD

source_corrections = get_source_corrections(window_days=4, post_correction_free=True).query('prior_corrections==0').reset_index(drop=True)
print(source_corrections.shape)
source_corrections.head()

### Original post daily [-4,4]

In [ ]:
ts_posts = pd.read_parquet('data/ts_posts_daily.parquet.gzip').query('-4<=days_from_display<=4').reset_index(drop=True)
ts_posts = ts_posts.merge(source_corrections, on='tweet_id', how='inner')
print(ts_posts.shape)
ts_posts.head()

In [ ]:
stata_did = StataDiD(ts_posts, 'treated', 'tweet_id', 'days_from_display', -4, 4)

In [ ]:
stata_did.multi_did(reg='ppmlhdfe', dev='tweet_count', cluster='account_cluster')

In [ ]:
from utils import get_coefs
means_mp_atts, lls_mp_atts, hls_mp_atts = get_coefs(reg='ppmlhdfe', start_line=8, end_line=15, stat_type='z', irr_or=1, unit_group=True)

In [ ]:
stata_did.tw_did(reg='ppmlhdfe', dev='tweet_count', cluster='account_cluster')

In [ ]:
mean_agg_att, ll_agg_att, hl_agg_att = get_coefs(reg='ppmlhdfe', start_line=2, end_line=3, stat_type='z', irr_or=1, unit_group=True)

In [ ]:
from utils import att_plot
ylims = (-0.262, 0.462)
att_plot(-4, 4, 4, means_mp_atts, lls_mp_atts, hls_mp_atts, mean_agg_att, ll_agg_att, hl_agg_att, 'Daily', 'Days from note display', 'Estimated DiD effect', 'did_posts_daily', ylims=ylims, legend_ncol=1, legend_title='First correction')

### Original post daily [-4,4]
window = 8

In [ ]:
source_corrections = get_source_corrections(window_days=8, post_correction_free=True).query('prior_corrections==0').reset_index(drop=True)
ts_posts = pd.read_parquet('data/ts_posts_daily.parquet.gzip').query('-4<=days_from_display<=4').reset_index(drop=True)
ts_posts = ts_posts.merge(source_corrections, on='tweet_id', how='inner')
print(ts_posts.shape)
ts_posts.head()

In [ ]:
stata_did = StataDiD(ts_posts, 'treated', 'tweet_id', 'days_from_display', -4, 4)

In [ ]:
stata_did.multi_did(reg='ppmlhdfe', dev='tweet_count', cluster='account_cluster')

In [ ]:
means_mp_atts, lls_mp_atts, hls_mp_atts = get_coefs(reg='ppmlhdfe', start_line=8, end_line=15, stat_type='z', irr_or=1, unit_group=True)

In [ ]:
stata_did.tw_did(reg='ppmlhdfe', dev='tweet_count', cluster='account_cluster')

In [ ]:
mean_agg_att, ll_agg_att, hl_agg_att = get_coefs(reg='ppmlhdfe', start_line=2, end_line=3, stat_type='z', irr_or=1, unit_group=True)

In [ ]:
att_plot(-4, 4, 4, means_mp_atts, lls_mp_atts, hls_mp_atts, mean_agg_att, ll_agg_att, hl_agg_att, 'Daily', 'Days from note display', 'Estimated DiD effect', 'did_posts_daily_8d', ylims=ylims, legend_ncol=1, legend_title='First correction')

### Original post daily [-28,28]

In [ ]:
source_corrections = get_source_corrections(window_days=28, post_correction_free=True).query('prior_corrections==0').reset_index(drop=True)
ts_posts = pd.read_parquet('data/ts_posts_daily.parquet.gzip')
ts_posts = ts_posts.merge(source_corrections, on='tweet_id', how='inner')
print(ts_posts.shape)
ts_posts.head()

In [ ]:
stata_did = StataDiD(ts_posts, 'treated', 'tweet_id', 'days_from_display', -28, 28)

In [ ]:
stata_did.multi_did(reg='ppmlhdfe', dev='tweet_count', cluster='account_cluster')

In [ ]:
means_mp_atts, lls_mp_atts, hls_mp_atts = get_coefs(reg='ppmlhdfe', start_line=56, end_line=111, stat_type='z', irr_or=1, unit_group=True)

In [ ]:
from pystata import stata
stata.run('testparm treated_periods_pre*')

In [ ]:
stata.run('testparm treated_periods_post*')

In [ ]:
stata_did.tw_did(reg='ppmlhdfe', dev='tweet_count', cluster='account_cluster')

In [ ]:
mean_agg_att, ll_agg_att, hl_agg_att = get_coefs(reg='ppmlhdfe', start_line=2, end_line=3, stat_type='z', irr_or=1, unit_group=True)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from utils import FONT_SIZE, LW, MZ, vermilion, blue, soft_red

def att_plot(ts_start, ts_end, mp_means_att, mp_lls_att, mp_hls_att, x_label, y_label, output_name, ylims=None, fig_size=(20,5)):
    plt.style.use('default')
    plt.rcParams['font.size'] = FONT_SIZE
    fig_plot, ax_plot = plt.subplots(figsize=fig_size, layout='constrained')
    plt.axhline(y=0, linestyle='--', color='grey', linewidth=LW)
    ax_plot.fill_between(np.arange(ts_start+0.5, -0.5, 1), np.array(mp_lls_att[:-ts_start-1]), np.array(mp_hls_att[:-ts_start-1]), alpha=0.25, color=vermilion)
    plt.errorbar(np.arange(ts_start+0.5, -0.5, 1), np.array(mp_means_att[:-ts_start-1]), yerr=0,
                 marker='', linestyle='-', color=vermilion, linewidth=LW)
    ax_plot.fill_between(np.arange(0.5, ts_end+0.5, 1), np.array(mp_lls_att[-ts_start-1:]), np.array(mp_hls_att[-ts_start-1:]), alpha=0.25, color=vermilion)
    plt.errorbar(np.arange(0.5, ts_end+0.5, 1), np.array(mp_means_att[-ts_start-1:]), yerr=0,
                 marker='', linestyle='-', color=vermilion, linewidth=LW)
    plt.xticks(np.arange(ts_start, ts_end+1, 4),
               [f'$-${abs(i)}' for i in np.arange(ts_start, 0, 4)]+[i for i in np.arange(0, ts_end+1, 4)])
    plt.xlim(ts_start-1.5, ts_end+1.5)
    if ylims is not None:
        plt.ylim(ylims[0], ylims[1])
    plt.xlabel(x_label)
    plt.ylabel(y_label)
    plt.savefig(f'outputs/figures/{output_name}.pdf')
    plt.show()

In [ ]:
att_plot(-28, 28, means_mp_atts, lls_mp_atts, hls_mp_atts, 'Days from note display', 'Estimated DiD effect', 'did_posts_daily_long', ylims=ylims)

In [ ]:
def att_plot(ts_start, ts_end, mp_means_att, mp_lls_att, mp_hls_att, x_label, y_label, output_name, ylims=None):
    plt.style.use('default')
    plt.rcParams['font.size'] = FONT_SIZE
    fig_plot, ax_plot = plt.subplots(layout='constrained')
    plt.axhline(y=0, linestyle='--', color='grey', linewidth=LW)
    ax_plot.fill_between(np.arange(ts_start+0.5, -0.5, 1), np.array(mp_lls_att[:-ts_start-1]), np.array(mp_hls_att[:-ts_start-1]), alpha=0.25, color=vermilion)
    plt.errorbar(np.arange(ts_start+0.5, -0.5, 1), np.array(mp_means_att[:-ts_start-1]), yerr=0,
                 marker='', linestyle='-', color=vermilion, linewidth=LW)
    ax_plot.fill_between(np.arange(0.5, ts_end+0.5, 1), np.array(mp_lls_att[-ts_start-1:]), np.array(mp_hls_att[-ts_start-1:]), alpha=0.25, color=vermilion)
    plt.errorbar(np.arange(0.5, ts_end+0.5, 1), np.array(mp_means_att[-ts_start-1:]), yerr=0,
                 marker='', linestyle='-', color=vermilion, linewidth=LW)
    plt.xticks([-28, -21, -14, -7, 0, 7, 14, 21, 28], ['-28', '-21', '-14', '-7', 0, 7, 14, 21, 28])
    plt.xlim(ts_start-1.9, ts_end+1.9)
    if ylims is not None:
        plt.ylim(ylims[0], ylims[1])
    plt.xlabel(x_label)
    plt.ylabel(y_label)
    plt.savefig(f'outputs/figures/{output_name}.pdf')
    plt.show()

att_plot(-28, 28, means_mp_atts, lls_mp_atts, hls_mp_atts, 'Days from note display', 'Estimated DiD effect', 'did_posts_daily_long_small', ylims=ylims)

In [ ]:
# Capture the current event-study sample before another regression replaces e(sample).
# Preserve the Stata data so subsequent coefficient extraction is unchanged.
stata.run('preserve', echo=False)
try:
    stata.run('quietly keep if e(sample)', echo=False)
    stata.run('keep tweet_id days_from_display treated tweet_count', echo=False)
    levels_sample_28d = stata.pdataframe_from_data()
finally:
    stata.run('restore', echo=False)

In [ ]:
from utils import SEED

def ts_count_plot(data, ts_start, ts_end, tcol, x_label, y_label, output_name, cip=95, ylims=(2.5, 13.5), fig_size=(20,5)):
    plt.style.use('default')
    plt.rcParams['font.size'] = FONT_SIZE
    plt.subplots(figsize=fig_size, layout='constrained')
    sns.lineplot(data[data['treated']==1], x=tcol, y='tweet_count', label='Treatment', color=soft_red, linewidth=LW, errorbar=('ci', cip), seed=SEED)
    sns.lineplot(data[data['treated']==0], x=tcol, y='tweet_count', label='Control', color=blue, linewidth=LW, linestyle='--', errorbar=('ci', cip), seed=SEED)
    plt.xticks(np.arange(ts_start, ts_end+1, 4),
               [f'$-${abs(i)}' for i in np.arange(ts_start, 0, 4)]+[i for i in np.arange(0, ts_end+1, 4)])
    plt.xlim(ts_start-1.5, ts_end+1.5)
    if ylims is not None:
        plt.ylim(ylims[0], ylims[1])
    plt.xlabel(x_label)
    plt.ylabel(y_label)
    plt.savefig(f'outputs/figures/{output_name}.pdf')
    plt.show()

In [ ]:
ts_count_plot(levels_sample_28d, -28, 28, tcol='days_from_display', x_label='Days from note display', y_label='Post count', output_name='ts_posts_daily')

### Synthetic DiD

In [ ]:
stata.run(
    f"""
    sdid tweet_count unitId days_from_display treated_after, ///
        vce(bootstrap) cluster(account_cluster) ///
        reps(100) seed({SEED}) graph ///
        g2_opt(xtitle("Days from note display") ytitle("Post count") xlabel(-28(4)28) title("") graphregion(color(white)))
    """
)

In [ ]:
stata.run(f"graph export outputs/figures/sdid_posts_daily.pdf, replace")